In [ ]:
import re
import json
from bs4 import BeautifulSoup

def parse_svoya_igra(html_content, year=1994):
    soup = BeautifulSoup(html_content, 'html.parser')
    
    # Удаляем сноски [1], [2] и т.д.
    for sup in soup.find_all('sup'):
        sup.decompose()
    
    questions = []
    current_round = 0
    
    # Собираем все значимые элементы
    elements = soup.find_all(['h3', 'h4', 'p'])
    
    i = 0
    while i < len(elements):
        el = elements[i]
        
        # 1. Определение раунда
        if el.name == 'h3':
            text = el.get_text().strip()
            if "Синий раунд" in text: current_round = 1
            elif "Красный раунд" in text: current_round = 2
            elif "Финальный раунд" in text: current_round = "Final"
            i += 1
            continue

        # 2. Обработка вопроса (тема и цена)
        if el.name == 'h4' and current_round != "Final":
            title = el.get_text().strip()
            match = re.search(r'^(.*?)\s*\((\d+)\)', title)
            if match:
                topic = match.group(1)
                price = int(match.group(2))
                
                question_parts = []
                correct_answer = ""
                players_answers = []
                
                # Собираем содержимое после заголовка h4
                j = i + 1
                while j < len(elements) and elements[j].name == 'p':
                    p_tag = elements[j]
                    raw_text = p_tag.get_text().strip()
                    
                    # Проверяем, это лог (ответы) или еще текст вопроса
                    if any(x in raw_text for x in ["Отвечает", "Играет", "Правильный ответ:"]):
                        # Разбиваем абзац на строки, сохраняя структуру <br/>
                        lines = [line.strip() for line in p_tag.get_text(separator="\n").split("\n") if line.strip()]
                        
                        was_player_called = False # Флаг: была ли строка "Отвечает ..."
                        
                        for line in lines:
                            # 1. Если строка про то, кто отвечает
                            if line.startswith("Отвечает") or line.startswith("Играет"):
                                was_player_called = True
                                continue
                            
                            # 2. Если это неправильный ответ игрока
                            if "Ответ игрока:" in line:
                                player_val = line.split(":", 1)[1].strip()
                                players_answers.append(player_val)
                                was_player_called = False # Сбрасываем флаг
                            
                            # 3. Если это правильный ответ
                            elif "Правильный ответ:" in line:
                                print(line)
                                correct_val = line.split(":", 1)[1].strip()
                                correct_answer = correct_val
                                
                                # ЛОГИКА: если перед этим был вызов игрока и НЕ БЫЛО строки "Ответ игрока",
                                # значит игрок сразу ответил правильно.
                                if was_player_called:
                                    players_answers.append(correct_val)
                                
                                was_player_called = False
                            
                            # Игнорируем ставки и прочее
                            elif "Ставка" in line:
                                pass
                            else:
                                was_player_called = False

                        # Если нашли правильный ответ, то вопрос закончен
                        if correct_answer:
                            j += 1
                            break
                    else:
                        # Накопление текста вопроса (игнорируем список тем в начале раунда)
                        if "Темы:" not in raw_text:
                            question_parts.append(raw_text)
                    j += 1
                
                questions.append({
                    "year": year,
                    "price": price,
                    "round": current_round,
                    "topic": topic,
                    "question": " ".join(question_parts).strip(),
                    "answer": correct_answer,
                    "players_answers": players_answers
                })
                i = j - 1
        
        # 3. Финальный раунд
        elif current_round == "Final" and el.name == 'p' and "Тема:" in el.get_text():
            topic = el.get_text().replace("Тема:", "").strip()
            i += 1
            question_text = elements[i].get_text(strip=True) if i < len(elements) else ""
            
            correct_answer = ""
            players_answers = []
            
            j = i + 1
            while j < len(elements) and current_round == "Final":
                p_tag = elements[j]
                p_text = p_tag.get_text().strip()
                
                if "Правильный ответ:" in p_text:
                    correct_answer = p_text.split(":", 1)[1].strip()
                    break
                elif "Ответ" in p_text and ":" in p_text:
                    # В финале формат: "Ответ Имя: Текст [Ставка]"
                    # Берем часть после имени до слова Ставка
                    ans_part = p_text.split(":", 1)[1]
                    ans_clean = ans_part.split("Ставка")[0].strip()
                    players_answers.append(ans_clean)
                j += 1
            
            questions.append({
                "year": year,
                "price": 0,
                "round": "Final",
                "topic": topic,
                "question": question_text,
                "answer": correct_answer,
                "players_answers": players_answers
            })
            break
            
        i += 1
    return questions

In [ ]:
import requests

url = "https://gameshows.ru/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%9E%D0%B1%D0%B7%D0%BE%D1%80_%D0%B2%D1%8B%D0%BF%D1%83%D1%81%D0%BA%D0%B0_1994-04-07)"

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
    'Accept-Language': 'ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7',
}

try:
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status() # Выдаст ошибку, если статус не 200
    response.encoding = 'utf-8'  # Явно указываем кодировку для кириллицы
    
    html_content = response.text
    print("Успешно загружено!")
    
except requests.exceptions.RequestException as e:
    print(f"Ошибка при загрузке: {e}")

Успешно загружено!


In [15]:
json_data = parse_svoya_igra(html_content, 1994)
print(json.dumps(json_data, ensure_ascii=False, indent=4))

Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
Правильный ответ:
[
    {
        "year": 1994,
        "price": 30,
        "round": 1,
        "topic": "В мире животных",
        "question": "В Англии

In [19]:
import re
import json
import requests
from bs4 import BeautifulSoup

def parse_svoya_igra_fixed(url, year=1994):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.encoding = 'utf-8'
        if response.status_code != 200:
            print(f"Ошибка доступа: {response.status_code}")
            return []
        html_content = response.text
    except Exception as e:
        print(f"Ошибка сети: {e}")
        return []

    soup = BeautifulSoup(html_content, 'html.parser')
    
    # Очистка от мусора
    for sup in soup.find_all('sup'): sup.decompose()
    
    content = soup.find('div', class_='mw-parser-output')
    if not content:
        print("Контент не найден")
        return []

    questions = []
    current_round = 0
    
    # Ищем ВСЕ заголовки и абзацы внутри контента в порядке их появления
    # Это решает проблему вложенности в div.mw-heading
    all_elements = content.find_all(['h3', 'h4', 'p'])
    
    i = 0
    while i < len(all_elements):
        el = all_elements[i]
        # Заменяем неразрывные пробелы \xa0 на обычные
        text = el.get_text().replace('\xa0', ' ').strip()

        # 1. Определение раунда
        if el.name == 'h3':
            if "Синий раунд" in text: current_round = 1
            elif "Красный раунд" in text: current_round = 2
            elif "Финальный раунд" in text: current_round = "Final"
            i += 1
            continue

        # 2. Обычные вопросы
        if el.name == 'h4' and current_round != "Final":
            # Регулярка для темы и цены
            match = re.search(r'(.+?)\s*\((\d+)\)', text)
            if match:
                topic = match.group(1).strip()
                price = int(match.group(2))
                
                q_text = ""
                correct_ans = ""
                p_answers = []
                
                # Собираем все <p> после h4 до следующего заголовка
                j = i + 1
                while j < len(all_elements) and all_elements[j].name == 'p':
                    p_tag = all_elements[j]
                    p_text = p_tag.get_text(" ", strip=True).replace('\xa0', ' ')
                    
                    if any(x in p_text for x in ["Отвечает", "Играет", "Правильный ответ"]):
                        # Логика извлечения ответов через регулярное разделение
                        # Мы ищем все вхождения меток и текст после них
                        parts = re.split(r'(Отвечает|Играет|Ответ игрока:|Правильный ответ:)', p_text)
                        
                        was_called = False
                        # В списке parts: [текст_до, метка, текст_после, метка, текст_после...]
                        for k in range(1, len(parts), 2):
                            label = parts[k]
                            val = parts[k+1].strip().rstrip('.').strip()
                            
                            if label in ["Отвечает", "Играет"]:
                                was_called = True
                            elif label == "Ответ игрока:":
                                p_answers.append(val)
                                was_called = False
                            elif label == "Правильный ответ:":
                                correct_ans = val
                                if was_called:
                                    p_answers.append(val)
                                was_called = False
                    else:
                        # Накопление текста вопроса
                        if "Темы:" not in p_text and p_text:
                            q_text += " " + p_text
                    j += 1
                
                questions.append({
                    "year": year,
                    "price": price,
                    "round": current_round,
                    "topic": topic,
                    "question": q_text.strip(),
                    "answer": correct_ans,
                    "players_answers": p_answers
                })
                i = j - 1 # Перескакиваем обработанные абзацы
        
        # 3. Финальный раунд
        elif current_round == "Final" and el.name == 'p' and "Тема:" in text:
            topic = text.replace("Тема:", "").strip()
            i += 1
            # Следующий p - вопрос
            q_text = all_elements[i].get_text(strip=True) if i < len(all_elements) else ""
            
            ans = ""
            p_ans = []
            
            j = i + 1
            while j < len(all_elements) and all_elements[j].name == 'p':
                p_text = all_elements[j].get_text(" ", strip=True).replace('\xa0', ' ')
                if "Правильный ответ:" in p_text:
                    ans = p_text.split(":", 1)[1].strip()
                    break
                elif "Ответ" in p_text and ":" in p_text:
                    # Извлекаем текст до слова "Ставка"
                    val = p_text.split(":", 1)[1].split("Ставка")[0].strip()
                    p_ans.append(val)
                j += 1
            
            questions.append({
                "year": year,
                "price": 0,
                "round": "Final",
                "topic": topic,
                "question": q_text,
                "answer": ans,
                "players_answers": p_ans
            })
            break # Финал обычно в конце
            
        i += 1
    return questions

# Проверка
url = "https://gameshows.ru/wiki/%D0%A1%D0%B2%D0%BE%D1%8F_%D0%B8%D0%B3%D1%80%D0%B0_(%D0%9E%D0%B1%D0%B7%D0%BE%D1%80_%D0%B2%D1%8B%D0%BF%D1%83%D1%81%D0%BA%D0%B0_1994-04-07)"
data = parse_svoya_igra_fixed(url, 1994)

if data:
    print(f"Questions parsed: {len(data)}")
    print(json.dumps(data, ensure_ascii=False, indent=4))
else:
    print("Questions list is empty. Check the connection or page structure.")

Questions parsed: 49
[
    {
        "year": 1994,
        "price": 30,
        "round": 1,
        "topic": "В мире животных",
        "question": "В Англии бетонные бункеры, оставшиеся после Второй мировой войны, начали переделывать в искусственные пещеры. Они предназначались для зимовки…",
        "answer": "Летучих мышей",
        "players_answers": [
            "Пчёл"
        ]
    },
    {
        "year": 1994,
        "price": 20,
        "round": 1,
        "topic": "В мире животных",
        "question": "Учёные считали, что эта рыба вымерла 70 млн лет назад, а она преспокойно плавает у южных берегов Африки.",
        "answer": "Латимерия",
        "players_answers": [
            "Латимерия"
        ]
    },
    {
        "year": 1994,
        "price": 20,
        "round": 1,
        "topic": "Книжный мир",
        "question": "Он писал: «Я — москвич! Сколь счастлив тот, кто может произносить это слово, вкладывая в него всего себя. Я — москвич!»",
        "answer": "Владимир 